# Customer Segmentation Using Unsupervised Clustering
**Category 03: Machine Learning | Project 05**

This walkthrough implements unsupervised customer segmentation to discover behavioral shopper archetypes and formulate targeted marketing playbooks.

### Key Highlights:
- **Standardization**: Why `StandardScaler` is required before calculating distance metrics.
- **Optimal Cluster Selection**: The **Elbow Method (Inertia)** and **Silhouette Score** sweep across $k \in [2, 10]$.
- **Algorithm Comparison**: Benchmarking **K-Means**, **Agglomerative Hierarchical Clustering**, and **DBSCAN** density clustering.
- **Dimensionality Reduction**: Visualizing clusters and centroids on a 2D **Principal Component Analysis (PCA)** plane.
- **Persona Profiling**: Translating numerical cluster centroids into intuitive consumer personas (e.g., Affluent VIPs, Trendsetters, Savers).
- **Actionable Strategies**: Marketing playbooks for each segment.
- **Inference & Scoring Engine**: Real-time scoring of new customer profiles.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to sys.path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data_loader import load_customer_data, CUSTOMER_FEATURES
from src.data_preprocessing import preprocess_customer_data, scale_customer_features, apply_pca_reduction
from src.cluster_models import train_all_clustering_models, save_clustering_artifact
from src.cluster_evaluation import (
    compute_elbow_and_silhouette_sweep,
    evaluate_clustering_metrics,
    profile_clusters,
    plot_clustering_diagnostics,
)
from src.segmenter import CustomerSegmenter

print("Environment initialized successfully.")

## 1. Data Ingestion & Behavioral Exploration
Loading retail shopper records containing Age, Annual Income, Spending Score, Frequency, and Average Basket Value.

In [ ]:
raw_df = load_customer_data()
print(f"Raw dataset shape: {raw_df.shape}")
raw_df.head(5)

## 2. Preprocessing & Feature Standardization
Deduplicating records and applying `StandardScaler` so that high-magnitude features (Income) do not artificially dominate smaller features (Spending Score).

In [ ]:
clean_df, report = preprocess_customer_data(raw_df)
print("Cleaning Report:", report)
X_scaled, scaler = scale_customer_features(clean_df)
X_pca, pca_model, exp_var = apply_pca_reduction(X_scaled)
print(f"2D PCA Explained Variance: {exp_var:.2%}")

## 3. Determining Optimal Clusters: Elbow & Silhouette Sweeps
Sweeping cluster count $k$ from 2 to 10 to inspect Inertia and Silhouette scores.

In [ ]:
sweep_df = compute_elbow_and_silhouette_sweep(X_scaled, k_range=range(2, 11))
display(sweep_df)

best_k = int(sweep_df.sort_values(by="Silhouette_Score", ascending=False).iloc[0]["k"])
print(f"Optimal cluster count by peak Silhouette score: k = {best_k}")

## 4. Multi-Algorithm Benchmarking: K-Means vs. Hierarchical vs. DBSCAN
Training partition and density clustering methods under identical standardized features.

In [ ]:
models_dict = train_all_clustering_models(X_scaled, n_clusters=best_k)
eval_records = [evaluate_clustering_metrics(X_scaled, res[1], name) for name, res in models_dict.items()]
pd.DataFrame(eval_records)

## 5. Cluster Profiling & Persona Translation
Translating mathematical centroids into business marketing personas.

In [ ]:
km_labels = models_dict["K-Means"][1]
profiles = profile_clusters(clean_df, km_labels)
for cid, p_info in sorted(profiles.items()):
    print(f"Cluster {cid}: {p_info['persona_name']} ({p_info['percentage']}%, N={p_info['customer_count']})")
    print(f"  - Description: {p_info['description']}")
    print(f"  - Strategy:    {p_info['marketing_strategy']}\n")

## 6. Model Serialization & Production Scoring
Saving the best clustering artifact and scoring a new customer profile.

In [ ]:
km_model = models_dict["K-Means"][0]
km_metrics = eval_records[0]
saved_path = save_clustering_artifact(
    model=km_model,
    scaler=scaler,
    algorithm_name="K-Means",
    n_clusters=best_k,
    metrics=km_metrics,
    cluster_profiles=profiles,
)
print(f"Clustering model serialized to: {saved_path}")

segmenter = CustomerSegmenter(model_path=saved_path)
sample_customer = {
    "Age": 38,
    "Annual_Income_k": 95.0,
    "Spending_Score": 85.0,
    "Purchase_Frequency": 32.0,
    "Average_Basket_Value": 210.0,
}
result = segmenter.score_single_customer(sample_customer)
for k, v in result.items():
    print(f"{k}: {v}")